In [1]:
%matplotlib inline
import matplotlib.pyplot as plt

import torch
import cv2
import pytesseract
import random

import numpy as np
import pandas as pd
import copy
import os, sys
import glob

import datetime as dt
import time
from timethis import timethis
import subprocess
import importlib
from common_params import parent_directory_images, parent_directory_url_csvs
from preprocess_data_fns import (
    create_image_files_df, get_predicted_categories_clip, categorize_and_plot, categories,
    crop_out_text, banner_text_box, preprocess_image_for_banner_text, body_style_dict

)


In [2]:
module_path='../01_collect_data/find_vehicle_image_urls.py'
spec = importlib.util.spec_from_file_location("find_vehicle_urls", module_path)
find_vehicle_urls = importlib.util.module_from_spec(spec)
sys.modules["find_vehicle_urls"] = find_vehicle_urls
spec.loader.exec_module(find_vehicle_urls)
find_vehicle_urls.get_vehicle_make_model_list

<function find_vehicle_urls.get_vehicle_make_model_list()>

In [3]:
output_filename_rules = f'{parent_directory_url_csvs}df_clip_categorize_rules.csv'
output_filename_croppped = f'{parent_directory_url_csvs}df_clip_categorize_cropped.csv'
output_filename_train_val_holdout = f'{parent_directory_url_csvs}df_train_val_hold.csv'

In [4]:

vehicle_make_model_list = find_vehicle_urls.get_vehicle_make_model_list()
# vehicle_make_model_list = get_vehicle_make_model_list()
vehicle_make_model_dict = {
    f'{x["make"]}_{x["model"]}' : x["body_style"]
    for x in vehicle_make_model_list
}
vehicle_make_model_dict
body_style_list = sorted(set(vehicle_make_model_dict.values()))
# body_style_dict = {x:i for i,x in enumerate(body_style_list)}
# body_style_dict = {'sedan': 0, 'sportscar': 1, 'suv': 2, 'truck': 3, 'van': 4, 'wagon': 5}
body_style_dict

{'sedan': 0, 'sportscar': 1, 'suv': 2, 'truck': 3, 'van': 4, 'wagon': 5}

In [5]:

df=pd.read_csv(output_filename_croppped)
print(df.shape)
print(df['is_exterior'].mean())
df = df[df['is_exterior']==1].reset_index(drop=True)
df['make_model'] = df.apply(
    lambda x: f'{x["make"]}_{x["model"]}' ,
    axis=1
)
df['body_style'] = df['make_model'].map(vehicle_make_model_dict)
df['body_style_index'] =df['body_style'].map(body_style_dict)

print(df.shape)
df.head()

(25913, 14)
0.33863311851194383
(8775, 17)


,filepath,filename,make,model,vehicle_id,is_car_exterior,top_category,probs_dict,probs_dict2,pct_exterior,is_exterior,reason,top3_categories,cropped_image_exists,make_model,body_style,body_style_index
0,/Users/levgolod/Projects/car_classifier/data/a...,376e2b73f7e04c0abb53dbed174a7bc8.jpg,bmw,3-series,701932606,1.0,car exterior,"{'car exterior': 0.8252363, 'advertisement': 0...","{'car exterior': 0.8252363, 'advertisement': 0...",0.83,1,pct_exterior>=70,"['car exterior', 'advertisement', 'gear select...",1,bmw_3-series,sedan,0
1,/Users/levgolod/Projects/car_classifier/data/a...,ac2d692d18f64c6b9eb4835d24bd42eb.jpg,bmw,3-series,701932606,1.0,car exterior,"{'car exterior': 0.7926975, 'advertisement': 0...","{'car exterior': 0.7926975, 'advertisement': 0...",0.79,1,pct_exterior>=70,"['car exterior', 'advertisement', 'gear select...",0,bmw_3-series,sedan,0
2,/Users/levgolod/Projects/car_classifier/data/a...,b4ade0a7c1c5426496acf7b3d45eb8d5.jpg,bmw,3-series,701932606,1.0,car exterior,"{'car exterior': 0.96773624, 'advertisement': ...","{'car exterior': 0.96773624, 'advertisement': ...",0.97,1,pct_exterior>=70,"['car exterior', 'advertisement', 'wheels clos...",0,bmw_3-series,sedan,0
3,/Users/levgolod/Projects/car_classifier/data/a...,e9ec18b6971c4b4a88bfdfd6da9cc331.jpg,bmw,3-series,701932606,1.0,car exterior,"{'car exterior': 0.8128994, 'advertisement': 0...","{'car exterior': 0.8128994, 'advertisement': 0...",0.81,1,pct_exterior>=70,NaN,0,bmw_3-series,sedan,0
4,/Users/levgolod/Projects/car_classifier/data/a...,50b55b74a0e04e8d9928d7bc48255f6f.jpg,bmw,3-series,704716526,1.0,car exterior,"{'car exterior': 0.9551495, 'car key': 0.02042...","{'car exterior': 0.9551495, 'car key': 0.02042...",0.96,1,pct_exterior>=70,NaN,0,bmw_3-series,sedan,0


In [6]:
# Volvo hatchback wagon, insufficient data so maybe skip  those?
print(df['body_style'].isnull().mean())
print(df['body_style_index'].isnull().mean())
# print(df.loc[df['body_style'].isnull(), 'make'].value_counts())
print(df.loc[df['body_style'].isnull(), 'make_model'].value_counts())
df = df[df['body_style'].notnull()].reset_index(drop=True)

0.0
0.0
Series([], Name: count, dtype: int64)


In [7]:
df['body_style'].value_counts(dropna=False).sort_index()
df[['body_style_index', 'body_style']].value_counts(dropna=False).sort_index()

body_style_index  body_style
0                 sedan         3960
1                 sportscar      431
2                 suv           1497
3                 truck         2311
4                 van            339
5                 wagon          237
Name: count, dtype: int64

In [8]:
df_vehicle_id = df[['vehicle_id']].drop_duplicates()
df_vehicle_id['partition'] = random.choices(
    population=['train','val','hold'],
    weights=[0.7, 0.15, 0.15],
    k=len(df_vehicle_id)
)
df_vehicle_id['partition'].value_counts(dropna=False,normalize=True)

partition
train    0.723753
hold     0.142219
val      0.134028
Name: proportion, dtype: float64

In [9]:
df['filepath_to_use'] = df['filepath']

for index, row in df.iterrows():
    if row['cropped_image_exists']:
        image_path = row['filepath']
        newfile = image_path.replace('.jpg','_cropped.jpg')
        if os.path.isfile(newfile):
            df.loc[index,'filepath_to_use'] = newfile

print(df['filepath_to_use'].str.contains('_cropped').sum())

538


In [10]:
df=df.merge(df_vehicle_id,on=['vehicle_id'],how='outer')
print(df['partition'].isnull().mean())

0.0


In [12]:
df2=df[['filepath','filename', 'filepath_to_use' ,'vehicle_id','partition', 'make','model','body_style','body_style_index']]
df2.head()

,filepath,filename,filepath_to_use,vehicle_id,partition,make,model,body_style,body_style_index
0,/Users/levgolod/Projects/car_classifier/data/a...,376e2b73f7e04c0abb53dbed174a7bc8.jpg,/Users/levgolod/Projects/car_classifier/data/a...,701932606,val,bmw,3-series,sedan,0
1,/Users/levgolod/Projects/car_classifier/data/a...,ac2d692d18f64c6b9eb4835d24bd42eb.jpg,/Users/levgolod/Projects/car_classifier/data/a...,701932606,val,bmw,3-series,sedan,0
2,/Users/levgolod/Projects/car_classifier/data/a...,b4ade0a7c1c5426496acf7b3d45eb8d5.jpg,/Users/levgolod/Projects/car_classifier/data/a...,701932606,val,bmw,3-series,sedan,0
3,/Users/levgolod/Projects/car_classifier/data/a...,e9ec18b6971c4b4a88bfdfd6da9cc331.jpg,/Users/levgolod/Projects/car_classifier/data/a...,701932606,val,bmw,3-series,sedan,0
4,/Users/levgolod/Projects/car_classifier/data/a...,50b55b74a0e04e8d9928d7bc48255f6f.jpg,/Users/levgolod/Projects/car_classifier/data/a...,704716526,train,bmw,3-series,sedan,0


In [13]:
df2.to_csv(output_filename_train_val_holdout, index=False)
print(output_filename_train_val_holdout)

/Users/levgolod/Projects/car_classifier/data/autotrader/vehicle_metadata/df_train_val_hold.csv


In [14]:
df2.shape

(8775, 9)